# TCGA (ESCC) + GTEx (Normal) — Xena ROC/AUC Analysis
### Esophageal Squamous Cell Carcinoma vs GTEx Normal Tissue
**Focus:** ROC & AUC for a predefined gene panel using UCSC Xena data

---
**Pipeline:**
1. Install & import
2. Target gene list
3. Load Xena data & assign labels
4. Build expression matrix
5. Preprocessing (NA filter only — data is already log2(norm_count + 1))
6. ROC/AUC per target gene
7. Individual ROC curves (separate figures)
8. Export results


## 1. Install & Import

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.metrics import roc_curve, auc

# ── Shared style ──
DPI     = 500
PALETTE = {"disease": "#E63946", "control": "#457B9D"}

plt.rcParams.update({
    "figure.dpi":        150,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         11,
})

OUTDIR = "./results_Xena_TCGA_GTEx"
os.makedirs(OUTDIR, exist_ok=True)

## 2. Target Gene List

In [ ]:
TARGET_GENES = ["gene_name"]

## 3. Load Xena Data & Assign Labels

In [ ]:
# Xena TSV: rows = samples, columns = genes (first col = sample ID)
xena = pd.read_csv("xenaDownload.tsv", sep="\t")
print(f"✓ Xena raw shape: {xena.shape}")

with open("tcga_sample_id_mrna.txt", "r") as f:
    tcga_ids = [line.strip() for line in f if line.strip()]
print(f"✓ TCGA ESCC sample IDs: {len(tcga_ids)}")

# Filter to GTEx + TCGA ESCC only
sample_col = xena.columns[0]
mask = xena[sample_col].str.startswith("GTEX") | xena[sample_col].isin(tcga_ids)
xena_clean = xena[mask].copy().reset_index(drop=True)

labels = xena_clean[sample_col].apply(
    lambda s: 0 if s.startswith("GTEX") else 1
)
labels.index = xena_clean[sample_col].values

print(f"\n0 = Control (GTEx)    : {(labels==0).sum()} samples")
print(f"1 = Disease (TCGA)    : {(labels==1).sum()} samples")
print(f"Total                 : {len(labels)} samples")

## 4. Build Expression Matrix

In [ ]:
# Drop sample ID column → genes as columns, samples as rows → transpose to genes × samples
sample_col = xena.columns[0]
expr = xena_clean.drop(columns=[sample_col]).T.copy()
expr.columns = xena_clean[sample_col].values
expr.index   = expr.index.astype(str).str.strip()

print(f"✓ Expression matrix: {expr.shape[0]} genes × {expr.shape[1]} samples")

## 5. Preprocessing — NA Filter Only

In [ ]:
# Xena data is already log2(norm_count + 1) — no log transform or quantile normalization needed
na_frac = expr.isnull().mean(axis=1)
expr    = expr.loc[na_frac <= 0.20]
expr    = expr.apply(lambda row: row.fillna(row.median()), axis=1)

print(f"After NA filter : {expr.shape[0]} genes")
print("Normalization   : skipped (Xena data is already log2(norm_count + 1))")

## 6. ROC/AUC per Target Gene

In [ ]:
y = labels.values

# Exact match only
gene_index_upper = {g.upper(): g for g in expr.index}

matched   = {}
unmatched = []
for gene in TARGET_GENES:
    key = gene.upper().strip()
    if key in gene_index_upper:
        matched[gene] = gene_index_upper[key]
    else:
        unmatched.append(gene)

print(f"✓ Matched   : {len(matched)} / {len(TARGET_GENES)}")
print(f"✗ Unmatched : {len(unmatched)}")
if unmatched:
    for g in unmatched: print(f"   {g}")

In [ ]:
roc_results = []

for gene, row_name in matched.items():
    x = expr.loc[row_name, labels.index].values.astype(float)

    fpr, tpr, _ = roc_curve(y, x)
    roc_auc     = auc(fpr, tpr)
    flipped = False
    if roc_auc < 0.5:
        fpr, tpr, _ = roc_curve(y, -x)
        roc_auc     = auc(fpr, tpr)
        flipped     = True

    log2fc = x[y==1].mean() - x[y==0].mean()
    _, pval = stats.ttest_ind(x[y==1], x[y==0], equal_var=False)

    roc_results.append({
        "Gene":      gene,
        "AUC":       roc_auc,
        "log2FC":    log2fc,
        "p_value":   pval,
        "direction": "Up" if log2fc > 0 else "Down",
        "flipped":   flipped,
        "fpr":       fpr,
        "tpr":       tpr,
    })

results_df = pd.DataFrame(
    [{k:v for k,v in r.items() if k not in ["fpr","tpr"]} for r in roc_results]
).sort_values("AUC", ascending=False).reset_index(drop=True)

print(f"✓ ROC/AUC computed for {len(results_df)} genes\n")
results_df[["Gene","AUC","log2FC","p_value","direction"]]

## 7. Individual ROC Curves (Separate Figures)

In [ ]:
for res in sorted(roc_results, key=lambda r: r["AUC"], reverse=True):
    fig, ax = plt.subplots(figsize=(4, 4))
    color = PALETTE["disease"] if res["log2FC"] > 0 else PALETTE["control"]
    ax.plot(res["fpr"], res["tpr"], lw=2, color=color)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4)
    ax.set_title(res["Gene"], fontsize=10, fontweight="bold")
    ax.text(0.97, 0.07, f"AUC={res['AUC']:.3f}", transform=ax.transAxes,
            ha="right", fontsize=9, color=color,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.8))
    ax.set_xlabel("False Positive Rate", fontsize=8)
    ax.set_ylabel("True Positive Rate", fontsize=8)
    ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1])
    plt.tight_layout()
    safe_name = res["Gene"].replace("/", "_")
    for ext in ("png", "tiff"):
        plt.savefig(f"{OUTDIR}/ROC_{safe_name}.{ext}", dpi=DPI, bbox_inches="tight")
    plt.close()

print(f"✓ Saved {len(roc_results)} individual ROC figures to {OUTDIR}")